# Reproduction: fork detection and base-trunk height

This notebook reproduces the evaluation reported in the manuscript: the
standalone fork-detection accuracy on manually corrected stems, the fork
detection for the two end-to-end pipelines, and the base-trunk height agreement
against the MLS reference.

All cells are checked in with their outputs, so the tables and figures are
visible without running anything. Rerunning requires the external tools and the
model weights described in the cell below, which covers the upstream setup.

## Upstream pipeline

<img src="figures/pipeline.png" width="600">

Upstream pipeline involve multiple steps and external tools:

The process starts with a raw ALS crop (e.g, ....las file in sample data)

**1. Denoise the raw ALS crop** (statistical outlier removal)

```bash
python upstream/denoise_sor.py sample_data/<tree>.las sample_data/<tree>_filtered.las
```

`--nb-neighbors` (default 6) sets how many neighbours estimate each point's
local mean distance; `--std-ratio` (default 2.0) is the cut-off in standard
deviations, with lower values removing more points.

**2. Instance Segmentation with ForestFormer3D**


Pipeline 1 — FOR-instanceV2 weights (public)
Follow the repository for run instructions

Pipeline 2 — FGI-EMIT weights (on request)
Contact the authors for code and model weights



```

**2. Instance segmentation** with ForestFormer3D:

```bash
# Pipeline 1 — FOR-instanceV2 weights (public)
<forestformer3d command, FOR-instanceV2 weights>

# Pipeline 2 — FGI-EMIT weights (on request)
<forestformer3d command, FGI-EMIT weights>
```

**3. Select the centre instance** for each plot (one tree per crop is evaluated):

```bash
<centre-instance selection command>
```

**4. Wood-leaf classification.** Pipeline 1 obtains this from ForestFormer3D in
step 2, so no separate step is needed. Pipeline 2 runs ResNet16FPN:

```bash
<resnet16fpn command>
```

**5. Prepare AdTree input** (keep stem points, write `.xyz`):

```bash
<stem-extraction command>
```

**6. Skeletonisation** with AdTree (one `<name>_skeleton.ply` per tree):

```bash
<adtree command>
```

Step 7, fork detection, is the evaluation reproduced below.

## Evaluation

### Setup

`config` puts the reproduction modules (`runner`, `reference`, `metrics`) on
the path and re-exports the fork-detection parameters from the main tool, so
the same thresholds drive both the tool and this notebook.

In [4]:
import os
from pathlib import Path
import sys
import warnings

# To regenerate from the dataset, set this before importing config, then Run All.
# Leave commented to read the checked-in outputs.
# os.environ["EVO_DATA_ROOT"] = "/path/to/EvoForkedTrees"


os.environ["EVO_DATA_ROOT"] = "/home/arun/PointClouds/SilvaFennica/code/test_data"

HERE = Path.cwd()
sys.path.insert(0, str(HERE))


import config
from runner import run_dataset
from reference import load_mls_reference
from metrics import (
    fork_detection_table, print_fork_detection,
    funnel_row, print_funnel_row,
    pair_estimates_with_reference, height_metrics, print_height_metrics,
)

print(config.EVO_DATA_ROOT)

/home/arun/PointClouds/SilvaFennica/code/test_data


### 1. Standalone fork detection

Fork/non-fork accuracy on the manually corrected stems. This isolates the
detector from upstream segmentation error and is the upper bound on detector
quality.

In [6]:
warnings.filterwarnings("ignore", message="duplicate XYZ merged")

forked   = run_dataset("manual_forked", force_rerun=True)
unforked = run_dataset("manual_unforked", force_rerun=True)

standalone = fork_detection_table(forked + unforked)
print_fork_detection(standalone, title="Manually corrected")

[manual_forked] running 175 skeletons from /home/arun/PointClouds/SilvaFennica/code/test_data/ManuallyCorrectedStems/Forked_ALS_AdTree_Output
  [10/175]  last=ALS_gridID_10_7_plotID_na_treeID_317_skeleton.ply  status=ok
  [20/175]  last=ALS_gridID_12_17_plotID_na_treeID_33_skeleton.ply  status=ok
  [30/175]  last=ALS_gridID_14_19_plotID_na_treeID_96_skeleton.ply  status=ok
  [40/175]  last=ALS_gridID_16_10_plotID_na_treeID_39_skeleton.ply  status=ok
  [50/175]  last=ALS_gridID_17_2_plotID_na_treeID_126_skeleton.ply  status=ok
  [60/175]  last=ALS_gridID_18_14_plotID_na_treeID_188_skeleton.ply  status=ok
  [70/175]  last=ALS_gridID_18_18_plotID_na_treeID_151_skeleton.ply  status=ok
  [80/175]  last=ALS_gridID_19_16_plotID_5_treeID_4_3_skeleton.ply  status=ok
  [90/175]  last=ALS_gridID_19_17_plotID_5_treeID_7_skeleton.ply  status=ok
  [100/175]  last=ALS_gridID_19_18_plotID_na_treeID_147_skeleton.ply  status=ok
  [110/175]  last=ALS_gridID_19_8_plotID_3_treeID_6_skeleton.ply  status=ok


### 2. End-to-end pipelines

For each end-to-end pipeline, this traces how many trees survive the instance
and semantic stages, and how many of the survivors are then given the correct
fork/non-fork label. Percentages are taken against `config.TOTAL_FORKED` and
`config.TOTAL_UNFORKED`

In [ ]:
PIPELINES = {
    "ForestFormer3D (FOR-instanceV2)": {
        "inst": ("p1_INST_forked", "p1_INST_unforked"),
        "sem":  ("p1_SEM_forked",  "p1_SEM_unforked"),
    },
    "ForestFormer3D (FGI-EMIT) + ResNet16FPN": {
        "inst": ("p2_INST_forked", "p2_INST_unforked"),
        "sem":  ("p2_SEM_forked",  "p2_SEM_unforked"),
    },
}

def stage_table(forked_ds, unforked_ds):
    return fork_detection_table(run_dataset(forked_ds) + run_dataset(unforked_ds))

for pipeline, stages in PIPELINES.items():
    inst_tab = stage_table(*stages["inst"])
    sem_tab  = stage_table(*stages["sem"])
    print(f"\n=== {pipeline} ===")
    for cls, total in (("forked", config.TOTAL_FORKED),
                       ("unforked", config.TOTAL_UNFORKED)):
        row = funnel_row(cls, inst_tab, sem_tab, total)
        print_funnel_row(row, pipeline=pipeline)

### 3. Base-trunk height vs MLS reference

For each setting, restrict to trees with an MLS reference, keep those detected
as forked, pair the predicted base-trunk height (`height_minmax`) against the
reference, and report RMSE, bias, MAE, R^2 and relative RMSE.

In [ ]:
lookup, mls_heights, _ = load_mls_reference()

SETTINGS = {
    "Manually corrected":                      "manual_forked",
    "ForestFormer3D (FOR-instanceV2)":         "p1_INST_forked",
    "ForestFormer3D (FGI-EMIT) + ResNet16FPN": "p2_INST_forked",
}

paired_by_setting = {}
for setting, ds in SETTINGS.items():
    recs = run_dataset(ds)
    recs = [r for r in recs if lookup(r["name"]) is not None]
    paired = pair_estimates_with_reference(
        recs, lookup, mls_heights, height_field="height_minmax")
    print_height_metrics(height_metrics(paired), title=setting)
    paired_by_setting[setting] = paired

Predicted against reference base-trunk height, one panel per setting. The
dashed line is 1:1.

In [ ]:
import matplotlib.pyplot as plt

settings = [s for s, p in paired_by_setting.items() if p]
fig, axes = plt.subplots(1, len(settings),
                         figsize=(4 * len(settings), 4),
                         sharex=True, sharey=True)
if len(settings) == 1:
    axes = [axes]

for ax, setting in zip(axes, settings):
    paired = paired_by_setting[setting]
    pred = [p for _, p, _ in paired]
    ref  = [r for _, _, r in paired]
    ax.scatter(ref, pred, s=18, alpha=0.7)
    lo = min(min(ref), min(pred))
    hi = max(max(ref), max(pred))
    ax.plot([lo, hi], [lo, hi], "k--", linewidth=1)
    ax.set_title(setting, fontsize=9)
    ax.set_xlabel("MLS reference (m)")

axes[0].set_ylabel("Predicted (m)")
fig.tight_layout()
plt.show()